# Phân tích mô tả và khám phá dữ liệu Olist

Notebook bản gọn: dữ liệu CSV đặt trong thư mục `data/`. Không cần Google Drive.


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats

# 1. Thiết lập thư mục làm việc
# Dữ liệu đầu vào nằm cùng thư mục với notebook
RAW_DIR = "./data"

# Thư mục lưu kết quả EDA
OUT_DIR = "./01_eda_output"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

print(f"Thư mục dữ liệu: {os.path.abspath(RAW_DIR)}")
print(f"Thư mục kết quả: {os.path.abspath(OUT_DIR)}")


## 1. Đọc dữ liệu và chuẩn hóa kiểu dữ liệu

Đọc 9 bảng Olist và chuyển các trường thời gian sang `datetime`.


In [ ]:
# 2. Đọc 9 file CSV
raw_files = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'translations': 'product_category_name_translation.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'geo': 'olist_geolocation_dataset.csv'
}

data = {k: pd.read_csv(os.path.join(RAW_DIR, v)) for k, v in raw_files.items()}
for k, v in data.items():
    print(f"Loaded: {raw_files[k]:<42} | Shape: {v.shape}")

# Chuẩn hóa kiểu Datetime
datetime_cols = {
    'orders': ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'],
    'order_items': ['shipping_limit_date'],
    'reviews': ['review_creation_date', 'review_answer_timestamp']
}
for tbl, cols in datetime_cols.items():
    if tbl in data:
        for col in cols:
            data[tbl][col] = pd.to_datetime(data[tbl][col], errors='coerce')


## 2. Cấu trúc dữ liệu và phân loại thuộc tính

Tổng quan quan hệ giữa các bảng và phân loại các thuộc tính tiêu biểu.


In [ ]:
# 4.1.1 TỔNG QUAN QUAN HỆ, ĐƠN VỊ QUAN SÁT & PHÂN LOẠI THUỘC TÍNH

schema_summary = pd.DataFrame([
    {"Bảng": "orders", "Primary Key (PK)": "order_id", "Foreign Key (FK)": "customer_id", "Đơn vị quan sát": "Đơn hàng (Order)"},
    {"Bảng": "order_items", "Primary Key (PK)": "(order_id, order_item_id)", "Foreign Key (FK)": "order_id, product_id, seller_id", "Đơn vị quan sát": "Mặt hàng trong đơn (Item)"},
    {"Bảng": "payments", "Primary Key (PK)": "(order_id, payment_sequential)", "Foreign Key (FK)": "order_id", "Đơn vị quan sát": "Lần thanh toán"},
    {"Bảng": "reviews", "Primary Key (PK)": "(review_id, order_id)", "Foreign Key (FK)": "order_id", "Đơn vị quan sát": "Lượt đánh giá"},
    {"Bảng": "customers", "Primary Key (PK)": "customer_id", "Foreign Key (FK)": "-", "Đơn vị quan sát": "Phiên khách hàng"},
    {"Bảng": "products", "Primary Key (PK)": "product_id", "Foreign Key (FK)": "product_category_name", "Đơn vị quan sát": "Sản phẩm"},
    {"Bảng": "sellers", "Primary Key (PK)": "seller_id", "Foreign Key (FK)": "-", "Đơn vị quan sát": "Người bán"},
    {"Bảng": "geo", "Primary Key (PK)": "Tổ hợp tọa độ", "Foreign Key (FK)": "-", "Đơn vị quan sát": "Mã bưu cục"},
    {"Bảng": "translations", "Primary Key (PK)": "product_category_name", "Foreign Key (FK)": "-", "Đơn vị quan sát": "Danh mục"}
])
print("\n==================== CẤU TRÚC VÀ QUAN HỆ CÁC BẢNG ====================")
display(schema_summary)
schema_summary.to_csv(os.path.join(TAB_DIR, "4_1_1_schema_relationship.csv"), index=False, encoding='utf-8-sig')

# BẢNG 4.1: Phân loại một số thuộc tính tiêu biểu
attribute_types = pd.DataFrame([
    {"Thuộc tính": "order_id, customer_id, seller_id", "Bảng gốc": "orders, items, sellers", "Loại biến": "Nominal", "Mô tả": "Mã định danh duy nhất cho thực thể/phiên"},
    {"Thuộc tính": "payment_type", "Bảng gốc": "order_payments", "Loại biến": "Nominal", "Mô tả": "Phương thức thanh toán"},
    {"Thuộc tính": "order_status", "Bảng gốc": "orders", "Loại biến": "Ordinal", "Mô tả": "Tiến trình trạng thái đơn hàng"},
    {"Thuộc tính": "review_score", "Bảng gốc": "order_reviews", "Loại biến": "Ordinal", "Mô tả": "Điểm mức độ hài lòng (1 đến 5 sao)"},
    {"Thuộc tính": "price, freight_value", "Bảng gốc": "order_items", "Loại biến": "Numeric", "Mô tả": "Giá bán sản phẩm và cước vận chuyển (BRL)"},
    {"Thuộc tính": "payment_value", "Bảng gốc": "order_payments", "Loại biến": "Numeric", "Mô tả": "Tổng giá trị thanh toán thực tế của đơn hàng"},
    {"Thuộc tính": "product_weight_g", "Bảng gốc": "products", "Loại biến": "Numeric", "Mô tả": "Trọng lượng thực tế của sản phẩm (gram)"},
    {"Thuộc tính": "order_purchase_timestamp", "Bảng gốc": "orders", "Loại biến": "Datetime", "Mô tả": "Thời điểm khách hàng bấm đặt mua hàng"}
])
print("\n==================== BẢNG 4.1: PHÂN LOẠI THUỘC TÍNH TIÊU BIỂU ====================")
display(attribute_types)
attribute_types.to_csv(os.path.join(TAB_DIR, "table_4_1_attribute_classification.csv"), index=False, encoding='utf-8-sig')


## 3. Chất lượng dữ liệu và tạo Master DataFrame

Kiểm tra dữ liệu khuyết, trùng lặp và tích hợp dữ liệu ở cấp độ mặt hàng.


In [ ]:
# 4.1.2 KHẢO SÁT CHẤT LƯỢNG DỮ LIỆU & PHÂN TÍCH MISSING VALUES

profiling_list = []
missing_field_list = []

for name, df in data.items():
    # Profiling tổng quan từng bảng
    profiling_list.append({
        'Bảng': name,
        'Số dòng': len(df),
        'Số cột': df.shape[1],
        'Dòng trùng lặp': df.duplicated().sum(),
        'Tổng ô NaN': df.isnull().sum().sum(),
        'Số cột có NaN': (df.isnull().sum() > 0).sum()
    })

    # Missing values chi tiết từng trường
    null_counts = df.isnull().sum()
    for col in df.columns:
        if null_counts[col] > 0:
            missing_field_list.append({
                'Bảng': name,
                'Trường': col,
                'Số lượng khuyết': null_counts[col],
                'Tỷ lệ (%)': round((null_counts[col] / len(df)) * 100, 2),
                'Kiểu dữ liệu': str(df[col].dtype)
            })

df_profiling = pd.DataFrame(profiling_list)
print("\n==================== KHẢO SÁT CHẤT LƯỢNG 9 BẢNG DỮ LIỆU ====================")
display(df_profiling)
df_profiling.to_csv(os.path.join(TAB_DIR, "4_1_2_data_profiling.csv"), index=False, encoding='utf-8-sig')

df_missing_fields = pd.DataFrame(missing_field_list).sort_values(by=['Bảng', 'Tỷ lệ (%)'], ascending=[True, False])
print("\n==================== PHÂN TÍCH MISSING VALUES THEO TỪNG TRƯỜNG ====================")
display(df_missing_fields)
df_missing_fields.to_csv(os.path.join(TAB_DIR, "2_1_missing_values_by_field.csv"), index=False, encoding='utf-8-sig')

# Chuẩn bị bảng tích hợp (Master DataFrame) cấp độ Item để thống kê
pay_agg = data['payments'].groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    primary_payment_type=('payment_type', lambda x: x.mode()[0] if not x.empty else np.nan)
).reset_index()

rev_agg = data['reviews'].groupby('order_id')['review_score'].mean().reset_index()

orders_df = data['orders'].copy()
orders_df['delivery_days'] = (orders_df['order_delivered_customer_date'] - orders_df['order_purchase_timestamp']).dt.total_seconds() / 86400

df_master = data['order_items'].merge(orders_df[orders_df['order_status'] == 'delivered'], on='order_id', how='inner')
df_master = df_master.merge(pay_agg, on='order_id', how='left')
df_master = df_master.merge(rev_agg, on='order_id', how='left')
df_master = df_master.merge(data['products'][['product_id', 'product_category_name', 'product_weight_g']], on='product_id', how='left')
df_master = df_master.merge(data['translations'], on='product_category_name', how='left')


## 4. Thống kê mô tả

Thống kê tần số, các biến định lượng và các biến định tính.


In [ ]:
# THỐNG KÊ ORDER_STATUS VÀ REVIEW_SCORE

def quick_freq_table(series, col_name):
    counts = series.value_counts(dropna=False)
    pcts = (series.value_counts(normalize=True, dropna=False) * 100).round(2)
    cum_pcts = pcts.cumsum().round(2)
    tbl = pd.DataFrame({
        'Thuộc tính': col_name,
        'Giá trị': counts.index.astype(str),
        'N': counts.values,
        'Tỷ lệ (%)': pcts.values,
        'Tích lũy (%)': cum_pcts.values
    })
    return tbl

tbl_status = quick_freq_table(data['orders']['order_status'], 'order_status')
tbl_review = quick_freq_table(data['reviews']['review_score'], 'review_score')
df_short_summary = pd.concat([tbl_status, tbl_review], ignore_index=True)

print("\n==================== THỐNG KÊ ORDER_STATUS VÀ REVIEW_SCORE ====================")
display(df_short_summary)
df_short_summary.to_csv(os.path.join(TAB_DIR, "short_status_and_review_summary.csv"), index=False, encoding='utf-8-sig')

# 4.1.3 THỐNG KÊ MÔ TẢ

# 1. Biến định lượng
numeric_cols = ['price', 'freight_value', 'delivery_days', 'total_payment', 'review_score']
num_stats = []

for c in numeric_cols:
    s = df_master[c].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)

    num_stats.append({
        'Biến': c,
        'N': int(len(s)),
        'Mean': s.mean(),
        'Median': s.median(),
        'Mode': s.mode()[0],
        'Std': s.std(),
        'Variance': s.var(),
        'Min': s.min(),
        'Max': s.max(),
        'Q1': q1,
        'Q3': q3,
        'IQR': q3 - q1,
        'Skewness': s.skew(),
        'Kurtosis': s.kurtosis()
    })

df_num_summary = pd.DataFrame(num_stats).round(2)
print("\n==================== THỐNG KÊ MÔ TẢ BIẾN ĐỊNH LƯỢNG ====================")
display(df_num_summary)
df_num_summary.to_csv(os.path.join(TAB_DIR, "4_1_3a_descriptive_numeric.csv"), index=False, encoding='utf-8-sig')

# 2. Biến định tính
def describe_cat(df, col, top_n=5):
    freq = df[col].value_counts(dropna=False).head(top_n)
    pct = (df[col].value_counts(normalize=True, dropna=False) * 100).head(top_n)
    return pd.DataFrame({
        'Frequency': freq,
        'Percentage (%)': pct.round(2),
        'Cumulative %': pct.cumsum().round(2)
    })

df_cat_pay = describe_cat(df_master, 'primary_payment_type')
print("\n==================== BIẾN ĐỊNH TÍNH: PAYMENT_TYPE ====================")
display(df_cat_pay)
df_cat_pay.to_csv(os.path.join(TAB_DIR, "4_1_3b_categorical_payment.csv"), encoding='utf-8-sig')

df_cat_prod = describe_cat(df_master, 'product_category_name_english', top_n=10)
print("\n==================== BIẾN ĐỊNH TÍNH: PRODUCT_CATEGORY ====================")
display(df_cat_prod)
df_cat_prod.to_csv(os.path.join(TAB_DIR, "4_1_3c_categorical_category.csv"), encoding='utf-8-sig')

import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


## 5. Phân phối đơn biến và trực quan hóa

Thiết lập biểu đồ, sau đó khảo sát histogram, KDE và boxplot.


In [ ]:
# Cấu hình giao diện biểu đồ
sns.set_theme(style='whitegrid', font_scale=0.95)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# HÌNH 1: 4.2.1 PHÂN TÍCH PHÂN PHỐI ĐƠN BIẾN (HISTOGRAM, KDE, BOXPLOT)

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
plt.subplots_adjust(hspace=0.45, wspace=0.3)

# 1. Price
p99_price = df_master['price'].quantile(0.99)
sns.histplot(df_master[df_master['price'] <= p99_price]['price'], kde=False, color='#2b5c8f', bins=35, ax=axes[0, 0])
axes[0, 0].axvline(df_master['price'].median(), color='red', linestyle='--', linewidth=1.5, label='Median')
axes[0, 0].axvline(df_master['price'].mean(), color='black', linestyle=':', linewidth=1.5, label='Mean')
axes[0, 0].set_title('Histogram: Price (<= 99%)', fontweight='bold')
axes[0, 0].set_xlabel('Price (BRL)')
axes[0, 0].set_ylabel('Số lượng')
axes[0, 0].legend(loc='upper right')

sns.kdeplot(df_master[df_master['price'] <= p99_price]['price'], color='#1f77b4', fill=True, alpha=0.3, ax=axes[0, 1])
axes[0, 1].set_title('KDE: Mật độ Price (Đuôi dài)', fontweight='bold')
axes[0, 1].set_xlabel('Price (BRL)')
axes[0, 1].set_ylabel('Mật độ')

sns.boxplot(x=df_master['price'], color='#a6c8e0', ax=axes[0, 2], fliersize=2)
axes[0, 2].set_title('Boxplot: Price & Outliers', fontweight='bold')
axes[0, 2].set_xlabel('Price (BRL)')

# 2. Freight Value
p99_freight = df_master['freight_value'].quantile(0.99)
sns.histplot(df_master[df_master['freight_value'] <= p99_freight]['freight_value'], kde=False, color='#2e7d32', bins=30, ax=axes[1, 0])
axes[1, 0].axvline(df_master['freight_value'].median(), color='red', linestyle='--', linewidth=1.5, label='Median')
axes[1, 0].axvline(df_master['freight_value'].mean(), color='black', linestyle=':', linewidth=1.5, label='Mean')
axes[1, 0].set_title('Histogram: Freight (<= 99%)', fontweight='bold')
axes[1, 0].set_xlabel('Freight Value (BRL)')
axes[1, 0].set_ylabel('Số lượng')
axes[1, 0].legend(loc='upper right')

sns.kdeplot(df_master[df_master['freight_value'] <= p99_freight]['freight_value'], color='#2e7d32', fill=True, alpha=0.3, ax=axes[1, 1])
axes[1, 1].set_title('KDE: Mật độ Freight (Đuôi dài)', fontweight='bold')
axes[1, 1].set_xlabel('Freight Value (BRL)')
axes[1, 1].set_ylabel('Mật độ')

sns.boxplot(x=df_master['freight_value'], color='#a5d6a7', ax=axes[1, 2], fliersize=2)
axes[1, 2].set_title('Boxplot: Freight & Outliers', fontweight='bold')
axes[1, 2].set_xlabel('Freight Value (BRL)')

# 3. Delivery Days
deliv_valid = df_master['delivery_days'].dropna()
p99_deliv = deliv_valid.quantile(0.99)
deliv_sub = deliv_valid[(deliv_valid >= 0) & (deliv_valid <= p99_deliv)]

sns.histplot(deliv_sub, kde=False, color='#d84315', bins=30, ax=axes[2, 0])
axes[2, 0].axvline(deliv_sub.median(), color='red', linestyle='--', linewidth=1.5, label='Median')
axes[2, 0].axvline(deliv_sub.mean(), color='black', linestyle=':', linewidth=1.5, label='Mean')
axes[2, 0].set_title('Histogram: Delivery Days (<= 99%)', fontweight='bold')
axes[2, 0].set_xlabel('Số ngày giao hàng')
axes[2, 0].set_ylabel('Số lượng')
axes[2, 0].legend(loc='upper right')

sns.kdeplot(deliv_sub, color='#d84315', fill=True, alpha=0.3, ax=axes[2, 1])
axes[2, 1].set_title('KDE: Mật độ Delivery Days (Đuôi dài)', fontweight='bold')
axes[2, 1].set_xlabel('Số ngày giao hàng')
axes[2, 1].set_ylabel('Mật độ')

sns.boxplot(x=df_master[df_master['delivery_days'] >= 0]['delivery_days'], color='#ffab91', ax=axes[2, 2], fliersize=2)
axes[2, 2].set_title('Boxplot: Delivery Days & Outliers', fontweight='bold')
axes[2, 2].set_xlabel('Số ngày giao hàng')

plt.suptitle('HÌNH 4.1: PHÂN TÍCH PHÂN PHỐI ĐƠN BIẾN & NGOẠI LỆ', fontsize=14, fontweight='bold', y=0.99)
plt.savefig(os.path.join(FIG_DIR, "fig4_1_univariate_distribution.png"), dpi=300, bbox_inches='tight')
plt.show()


## 6. Ngoại lệ và phân tích hai biến

Đánh giá outlier bằng quy tắc 1,5 × IQR và khảo sát các mối quan hệ hai biến.


In [ ]:
# Đếm và lưu bảng thống kê Outliers theo 1.5 * IQR
outlier_list = []
print("==================== ĐÁNH GIÁ MỨC ĐỘ NGOẠI LỆ (1.5 * IQR) ====================")
for col in ['price', 'freight_value', 'delivery_days']:
    s = df_master[col].dropna()
    if col == 'delivery_days':
        s = s[s >= 0]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = s[(s < lower_bound) | (s > upper_bound)]

    outlier_list.append({
        'Biến': col,
        'Ngưỡng dưới': round(lower_bound, 2),
        'Ngưỡng trên': round(upper_bound, 2),
        'Số lượng Outliers': len(outliers),
        'Tỷ lệ Outliers (%)': round(len(outliers) / len(s) * 100, 2)
    })
    print(f"- {col:<15}: {len(outliers):>6} dòng ({len(outliers)/len(s)*100:.2f}%) vượt ngưỡng trên [{upper_bound:.2f}]")

df_outliers = pd.DataFrame(outlier_list)
display(df_outliers)
df_outliers.to_csv(os.path.join(TAB_DIR, "4_2_1_outlier_summary.csv"), index=False, encoding='utf-8-sig')

# HÌNH 2: 4.2.2 PHÂN TÍCH MỐI QUAN HỆ HAI BIẾN (BIVARIATE ANALYSIS)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
plt.subplots_adjust(wspace=0.25)

# 1. Delivery Days vs Review Score (Chuẩn hóa về các mốc nguyên 1, 2, 3, 4, 5)
clean_rev = df_master.dropna(subset=['delivery_days', 'review_score']).copy()
clean_rev['review_score_int'] = clean_rev['review_score'].round().astype(int)
clean_rev = clean_rev[(clean_rev['delivery_days'] >= 0) & (clean_rev['delivery_days'] <= 45)]
clean_rev = clean_rev[clean_rev['review_score_int'].isin([1, 2, 3, 4, 5])]

sns.boxplot(
    x='review_score_int', y='delivery_days', hue='review_score_int', data=clean_rev, palette='Blues_r', ax=axes[0],
    showmeans=True, meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"red", "markersize": 5},
    fliersize=1.5, legend=False
)
axes[0].set_title('Delivery Days vs Review Score\n(Chấm đỏ: Giá trị Mean)', fontweight='bold')
axes[0].set_xlabel('Điểm đánh giá (Sao)')
axes[0].set_ylabel('Thời gian giao hàng (Ngày)')
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['1.0', '2.0', '3.0', '4.0', '5.0'])

# 2. Price vs Freight Value
sample_scatter = df_master[(df_master['price'] <= 600) & (df_master['freight_value'] <= 100)].sample(4000, random_state=42)
sns.scatterplot(x='price', y='freight_value', data=sample_scatter, alpha=0.25, color='#e65100', s=18, ax=axes[1])
sns.regplot(x='price', y='freight_value', data=sample_scatter, scatter=False, ax=axes[1], color='#0d47a1', line_kws={'linewidth': 1.8})
axes[1].set_title('Price vs Freight Value\n(Đường hồi quy xu hướng)', fontweight='bold')
axes[1].set_xlabel('Giá sản phẩm (BRL)')
axes[1].set_ylabel('Cước vận chuyển (BRL)')

# 3. Review Score trung bình theo Payment Type
clean_pay = df_master.dropna(subset=['primary_payment_type', 'review_score'])
clean_pay = clean_pay[clean_pay['primary_payment_type'].isin(['credit_card', 'boleto', 'voucher', 'debit_card'])]
review_by_pay = clean_pay.groupby('primary_payment_type')['review_score'].mean().reset_index()

barplot = sns.barplot(x='primary_payment_type', y='review_score', hue='primary_payment_type', data=review_by_pay, palette='viridis', ax=axes[2], legend=False)
axes[2].set_ylim(3.8, 4.4)
axes[2].set_title('Điểm Review trung bình theo Payment Type', fontweight='bold')
axes[2].set_xlabel('Phương thức thanh toán')
axes[2].set_ylabel('Review Score TB')
axes[2].tick_params(axis='x', rotation=15)

for p in barplot.patches:
    axes[2].annotate(f"{p.get_height():.2f}",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', xytext=(0, 3),
                     textcoords='offset points', fontweight='bold', fontsize=9.5)

plt.suptitle('HÌNH 4.2: PHÂN TÍCH MỐI QUAN HỆ HAI BIẾN', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(FIG_DIR, "fig4_2_bivariate_analysis.png"), dpi=300, bbox_inches='tight')
plt.show()


## 7. Tương quan Spearman

Tính và trực quan hóa ma trận tương quan hạng Spearman.


In [ ]:
# HÌNH 3: 4.2.2 MA TRẬN TƯƠNG QUAN ĐA BIẾN (SPEARMAN HEATMAP)

plt.figure(figsize=(9, 6.5))

corr_cols = ['price', 'freight_value', 'product_weight_g', 'delivery_days', 'total_payment', 'review_score']
corr_labels = ['Price', 'Freight', 'Weight', 'Delivery Days', 'Total Pay', 'Review']
corr_matrix = df_master[corr_cols].corr(method='spearman')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, vmax=1.0, vmin=-1.0, center=0,
    annot=True, fmt='.2f', square=True, linewidths=.6,
    xticklabels=corr_labels, yticklabels=corr_labels,
    annot_kws={"size": 10, "weight": "bold"}, cbar_kws={"shrink": .8}
)
plt.title('HÌNH 4.3: MA TRẬN TƯƠNG QUAN HẠNG SPEARMAN', fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.savefig(os.path.join(FIG_DIR, "fig4_3_correlation_heatmap.png"), dpi=300, bbox_inches='tight')
plt.show()

# Lưu bảng ma trận tương quan Spearman ra CSV
corr_matrix_export = corr_matrix.round(3)
corr_matrix_export.columns = corr_labels
corr_matrix_export.index = corr_labels
display(corr_matrix_export)
corr_matrix_export.to_csv(os.path.join(TAB_DIR, "4_2_2_spearman_correlation_matrix.csv"), encoding='utf-8-sig')


## Kết quả đầu ra

Các bảng CSV được lưu trong `01_eda_output/tables` và biểu đồ PNG trong `01_eda_output/figures`.
